> BM25 : 相关性 = 词的重要性IDF × 词频贡献TF × 文档长度归一化<br>
> 文档分词 → 统计 TF / DF / IDF → 对 Query 做 BM25 打分 → 排序 → Top-K


- BM25Okapi：	最经典、最常用的 BM25
- BM25L：	修正长文档可能被过度惩罚的问题
- BM25Plus：	在 BM25 基础上进一步增加一个正向补偿项

```
Query
  ↓
分词
  ↓
计算 Query 中每个词对每篇文档的 BM25 得分
  ↓
把各个词的得分累加
  ↓
文档排序
  ↓
Top-K
```

In [1]:
from rank_bm25 import (
    BM25Okapi,
    BM25L,
    BM25Plus,
)

In [3]:
corpus = [
    ["绝缘子", "发生", "自爆", "故障"],
    ["无人机", "发现", "绝缘子", "缺失"],
    ["变压器", "出现", "漏油", "故障"],
]

query = ["绝缘子", "故障"]

In [5]:
bm25_a = BM25Okapi(corpus)
score_a = bm25_a.get_scores(query)
print(f"{score_a = }")

score_a = array([0.15324769, 0.07662384, 0.07662384])


In [6]:
bm25_b = BM25L(corpus)
score_b = bm25_b.get_scores(query)
print(f"{score_b = }")

score_b = array([1.17500907, 0.58750454, 0.58750454])


In [7]:
bn25_c = BM25Plus(corpus)
score_c = bn25_c.get_scores(query)
print(f"{score_c = }")

score_c = array([2.77258872, 2.07944154, 2.07944154])


In [8]:
tops_a = bm25_a.get_top_n(query, corpus, n=2)
print(f"{tops_a = }")

tops_a = [['绝缘子', '发生', '自爆', '故障'], ['变压器', '出现', '漏油', '故障']]


In [9]:
tops_b = bm25_b.get_top_n(query, corpus, n=2)
print(f"{tops_b = }")

tops_b = [['绝缘子', '发生', '自爆', '故障'], ['变压器', '出现', '漏油', '故障']]


In [10]:
tops_c = bn25_c.get_top_n(query, corpus, n=2)
print(f"{tops_c = }")

tops_c = [['绝缘子', '发生', '自爆', '故障'], ['变压器', '出现', '漏油', '故障']]


In [ ]:
bm25_base = BM25Okapi(
    corpus,
    tokenizer=None,
    k1=1.5,  # 饱和速度 控制内部饱和函数，词频越高、重要性的增长越缓
    b=0.75,  # <---> 词频 / 文档长度
    epsilon=0.25,   # IDF平滑参数，避免IDF为负数的情况
    )
# 缺点：这个长度惩罚有时候下手太重了。
# 短文档、词更加集中、较高贡献、
# 长文档、词更加分散、较低贡献




In [ ]:
bm25_plus = BM25Plus(
    corpus,
    k1=1.5,
    b=0.75,
    delta=0.25,  # 对长度归一化之后的 TF 增加一个 lower bound，让长文档不要被压得太狠。
)
# TF -> BM25 normalization -> BM25 contribution + delta

In [ ]:
bm25_l = BM25L(
    corpus,
    k1=1.5,
    b=0.75,
    delta=0.5,  # 降低长度归一化对词频贡献压制过重的问题。  # chunks 长度不一致、变化大的情况下合适
)
# TF -> 长度归一化 -> ctd -> ctd + delta -> 饱和函数
